In [ ]:
import serial
import serial.tools.list_ports
import time
import cv2
import os
import urllib.request

# ==================================================
# CONFIGURATION & SETTINGS
# ==================================================
NUMBER_OF_DROPS = 24
PHOTO_FOLDER = "Fotos"
# This URL targets the internal Jupyter notebook server that is ALWAYS running inside your Flex
ROBOT_FLAG_URL = "http://192.168.40.25:48888/files/drop.txt"

ROI_Y1, ROI_Y2 = 350, 1080
ROI_X1, ROI_X2 = 450, 1250
ARDUINO_SERIAL = '03536373332351F03170'

# Connect to Arduino
ports = serial.tools.list_ports.comports()
arduino_port = None
for port in ports:
    if ARDUINO_SERIAL in str(port.serial_number):
        arduino_port = port.device

if arduino_port is None: 
    raise Exception("Hardware Missing: Arduino not detected.")
arduino = serial.Serial(arduino_port, 9600, timeout=1)
time.sleep(2)

tray_switch = {'foreward': b'a\n', 'backward': b'b\n', 'home': b'c\n'}

def wait_for_arduino():
    while True:
        if arduino.in_waiting > 0:
            response = arduino.readline().decode('utf-8').strip()
            if response == "DONE": 
                break

def TakePhoto(photo_number, suffix):
    if not os.path.exists(PHOTO_FOLDER): 
        os.makedirs(PHOTO_FOLDER)
    capture = cv2.VideoCapture(0, cv2.CAP_DSHOW)
    capture.set(cv2.CAP_PROP_FRAME_WIDTH, 1920)
    capture.set(cv2.CAP_PROP_FRAME_HEIGHT, 1080)
    capture.set(cv2.CAP_PROP_AUTOFOCUS, 0)
    capture.set(cv2.CAP_PROP_FOCUS, 140)
    time.sleep(0.5)
    for _ in range(5): 
        ret, frame = capture.read()
    capture.release()
    
    if not ret:
        print(f"⚠️ Camera error on drop {photo_number}")
        return
    crop = frame[ROI_Y1:ROI_Y2, ROI_X1:ROI_X2]
    filename = os.path.join(PHOTO_FOLDER, f"{photo_number}_{suffix}.png")
    cv2.imwrite(filename, crop)
    
    absolute_path = os.path.abspath(filename)
    print(f"📸 Saved -> {absolute_path}")
    
    if photo_number == 1: 
        os.startfile(os.path.abspath(PHOTO_FOLDER))

# ==================================================
# MAIN EXPERIMENT CONTROL LOOP
# ==================================================
print("\n=======================================================")
print("   HTTP CLIENT HANDSHAKE ACTIVE: STANDING BY           ")
print("=======================================================")
print("Calibrating rotational stage (HOME)...")
arduino.write(tray_switch['home'])
wait_for_arduino()

print("\n🚀 System Ready! Start your Opentrons Protocol run now.")

for i in range(1, NUMBER_OF_DROPS + 1):
    print(f"\n[DROPLET {i}/{NUMBER_OF_DROPS}] Polling robot for drop confirmation...")
    
    
    while True:
        try:
            with urllib.request.urlopen(ROBOT_FLAG_URL, timeout=2) as response:
                content = response.read().decode('utf-8')
                if "DROP_DONE" in content:
                    break
        except Exception:
            pass
        time.sleep(0.3)
        
    print(f"✅ Handshake verified from robot server for drop {i}!")
    print("⏳ Allowing droplet to settle before capturing image...")
    time.sleep(1)  
    
    # 2.Taking Picture
    TakePhoto(i, "droplet")
    time.sleep(0.5)
    
    # 3. Next Movement
    if i < NUMBER_OF_DROPS:
        print("Advancing rotational stage to next gel position...")
        arduino.write(tray_switch['foreward'])
        wait_for_arduino()
        
    print("Signaling robot to proceed...")
    
    # 4.Handshake waiting
    print("Waiting for robot to clear the 'DROP_DONE' flag before next turn...")
    while True:
        try:
            with urllib.request.urlopen(ROBOT_FLAG_URL, timeout=2) as response:
                content = response.read().decode('utf-8')
                if "DROP_DONE" not in content: # 
                    break
        except Exception:
            
            break
        time.sleep(0.4)

print("\nExperiment complete. Returning tray configuration home...")
arduino.write(tray_switch['home'])
wait_for_arduino()
arduino.close()
print("Execution cleanly terminated.")


   HTTP CLIENT HANDSHAKE ACTIVE: STANDING BY           
Calibrating rotational stage (HOME)...

🚀 System Ready! Start your Opentrons Protocol run now.

[DROPLET 1/24] Polling robot for drop confirmation...
✅ Handshake verified from robot server for drop 1!
⏳ Allowing droplet to settle before capturing image...
📸 Saved -> C:\Users\nahid.salimi\OneDrive\Fotos\1_droplet.png
Advancing rotational stage to next gel position...
Signaling robot to proceed...
Waiting for robot to clear the 'DROP_DONE' flag before next turn...

[DROPLET 2/24] Polling robot for drop confirmation...
